# Extract review aspects

This notebook is part of the public reproducibility workflow for the study.

Public workflow stage: 02  
Purpose: Extract sentence-level aspect terms with the formal i2 prompt.

Source notebook:
- `aspect_terms_extraction.ipynb` (formal prompt, batching, retries, resume, serialization)

Input: `data/work/sentence_parts/`. Default `mock` mode also requires `data/sample/mock_aspect_responses.jsonl` (provided later).  
Output: `data/work/aspect_responses/results_part_*.jsonl` and companion error/state files.  
`api` mode is opt-in and needs `DEEPSEEK_API_KEY` in the environment.


In [ ]:
from pathlib import Path
import sys

# Jupyter may start in the repository root or its notebooks directory.
_candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
_root = next((p for p in _candidates if (p / "src" / "project_paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory")
sys.path.insert(0, str(_root))
from src.project_paths import SAMPLE_DATA_DIR, WORK_DATA_DIR, CONFIG_DIR, project_path

try:
    import yaml
except ImportError as exc:
    raise ImportError("Install PyYAML to read config/config.example.yaml") from exc
with (CONFIG_DIR / "config.example.yaml").open(encoding="utf-8") as _f:
    config = yaml.safe_load(_f)
sample_dir = project_path(config["paths"]["sample_data"])
work_dir = project_path(config["paths"]["work_data"])

import json
import os
import random
import re
import time
import pandas as pd
from tqdm.auto import tqdm

sentence_dir = work_dir / "sentence_parts"
response_dir = work_dir / "aspect_responses"
mode = config["aspect_extraction"]["mode"]
if mode not in {"mock", "api"}:
    raise ValueError("aspect_extraction.mode must be 'mock' or 'api'")
if not sentence_dir.is_dir():
    raise FileNotFoundError(f"Sentence parts are missing: {sentence_dir}")
response_dir.mkdir(parents=True, exist_ok=True)
settings = config["aspect_extraction"]
MODEL = settings["model"]
TEMPERATURE = settings["temperature"]
MAX_TOKENS = settings["max_tokens"]
SEED = settings["seed"]
CHUNK_SIZE = int(settings["batch_size"])
MAX_FAIL_CHUNKS = int(settings["max_failed_chunks"])
SAVE_EVERY = int(settings["save_every"])
SLEEP_EACH_CALL = float(settings["sleep_each_call"])


## Formal prompt and response interface

Keys are global sentence indices within a review. The model must return a JSON object with string values; the parser keeps the source behavior of allowing partial success when at most one chunk fails.


In [ ]:
PROMPT = """Your task is to identify the aspect term(s) each sentence comments on.
Input: JSON {id: sentence}. Output: JSON {id: \"Aspect1, Aspect2\"}.

Rules:
1) Output 1–2 aspect terms (max 3 only if clearly necessary). Use short noun phrases.
2) Prefer a normalized 2-word form: <Topic> + <Type>.
   - Topic: method, data, result, experiment, figure, writing, organization, terminology, grammar, related work, evaluation, theory.
   - Type (choose the BEST match, do not default to one word): clarity, correctness, completeness, validity, robustness, design, description, reporting, comparison, consistency, novelty, reproducibility, limitation.
   Use clarity/interpretation ONLY if the sentence explicitly focuses on clarity/interpretation; otherwise choose a more specific Type.
3) IMPORTANT anti-template: Do NOT reuse wording from this prompt as a default template. Only output what is supported by the sentence.
4) Generic single words are allowed only if no specific 2-word aspect fits.
5) Output \"None\" if meaningless/citation-only/pure caption or ID reference (e.g., Fig. 2C, Table S1).

Format: valid JSON, keys unchanged, values are strings, no commas inside a single aspect term.
"""

def append_jsonl(path, row):
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(row, ensure_ascii=False) + "\n")

def load_done_reviews(path):
    done = set()
    if path.exists():
        with path.open(encoding="utf-8") as stream:
            for line in stream:
                try:
                    row = json.loads(line)
                    if row.get("review_id"):
                        done.add(str(row["review_id"]))
                except json.JSONDecodeError:
                    continue
    return done

def build_sentence_dict(frame):
    sentences = frame.sort_values("sent_idx", kind="mergesort")["sentence_clean"].tolist()
    return {str(index): sentence for index, sentence in enumerate(sentences)}

def safe_chat_completion(client, **kwargs):
    max_retries = int(settings["max_retries"])
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(**kwargs)
            if not response.choices or not response.choices[0].message.content:
                raise RuntimeError("empty_or_invalid_response_structure")
            return response
        except Exception as exc:
            message = str(exc).lower()
            retryable = any(token in message for token in
                            ("429", "rate limit", "timeout", "timed out", "connection",
                             "reset", "ssl", "502", "503", "504",
                             "empty_or_invalid_response_structure"))
            if not retryable and attempt >= 1:
                return None
            time.sleep((2 ** attempt) * 1.5 + random.random())
    return None


## Select an offline or explicit API response provider

Mock rows must contain `paper_id`, `review_id`, and `aspects` (a JSON object or its serialized string). The public sample is added in a later batch. No network client is constructed in default mode.


In [ ]:
if mode == "mock":
    mock_path = sample_dir / "mock_aspect_responses.jsonl"
    if not mock_path.is_file():
        raise FileNotFoundError(f"Mock responses are not available yet: {mock_path}")
    mock_responses = {}
    with mock_path.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                item = json.loads(line)
                mock_responses[(str(item["paper_id"]), str(item["review_id"]))] = item["aspects"]
    client = None
else:
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError("Explicit api mode requires DEEPSEEK_API_KEY")
    from openai import OpenAI
    client = OpenAI(base_url="https://api.deepseek.com/", api_key=api_key)

def request_batch(batch, paper_code, review_id):
    if mode == "mock":
        value = mock_responses.get((paper_code, review_id))
        if value is None:
            return None, "mock_response_missing"
        if isinstance(value, str):
            try:
                value = json.loads(value)
            except json.JSONDecodeError:
                return None, "invalid_json_response"
        if not isinstance(value, dict):
            return None, "invalid_json_response"
        subset = {str(key): value[str(key)] for key in batch if str(key) in value}
        return subset, None
    response = safe_chat_completion(
        client, model=MODEL,
        messages=[{"role": "system", "content": "You are an information extraction expert."},
                  {"role": "user", "content": PROMPT + "\n\n" + json.dumps(batch, ensure_ascii=False)}],
        temperature=TEMPERATURE, max_tokens=MAX_TOKENS, presence_penalty=0,
        seed=SEED, response_format={"type": "json_object"})
    if response is None:
        return None, "chat_completion_failed"
    try:
        value = json.loads(response.choices[0].message.content)
        return (value, None) if isinstance(value, dict) else (None, "invalid_json_response")
    except (AttributeError, IndexError, TypeError, json.JSONDecodeError):
        return None, "invalid_json_response"

def call_aspects_for_review(sent_dict, paper_code, review_id):
    keys = sorted(sent_dict, key=lambda key: int(key))
    merged, failed = {}, []
    for start in range(0, len(keys), CHUNK_SIZE):
        batch_keys = keys[start:start + CHUNK_SIZE]
        batch = {key: sent_dict[key] for key in batch_keys}
        value, error = request_batch(batch, paper_code, review_id)
        if error:
            failed.append(dict(start=start, size=len(batch_keys), error=error))
            if len(failed) > MAX_FAIL_CHUNKS:
                return None, f"chunk_failed_exceed_limit|fails={len(failed)}|info={failed[:2]}"
            continue
        merged.update({str(key): item for key, item in value.items()})
    if not merged:
        return None, "all_chunks_failed"
    warning = f"partial_success|fails={len(failed)}|missing={len(keys)-len(merged)}" if failed else None
    return json.dumps(merged, ensure_ascii=False), warning


## Process parts with append-only resume state

The source writes successful results as JSONL. Partial or failed reviews go to error files for explicit retry; repeated successful records are resolved in Notebook 03 by last-write-wins within each part.


In [ ]:
part_files = sorted(sentence_dir.glob("part-*.parquet"))
if not part_files:
    raise FileNotFoundError(f"No sentence-part parquet files were found in {sentence_dir}")
processed_total = 0
for part_path in part_files:
    match = re.fullmatch(r"part-(\d{5})", part_path.stem)
    if not match:
        continue
    index = match.group(1)
    results_path = response_dir / f"results_part_{index}.jsonl"
    errors_path = response_dir / f"errors_part_{index}.jsonl"
    state_path = response_dir / f"state_part_{index}.json"
    done = load_done_reviews(results_path)
    frame = pd.read_parquet(part_path, columns=["paper_code", "review_id", "sent_idx", "sentence_clean"])
    groups = frame.groupby(["paper_code", "review_id"], sort=False)
    for (paper_code, review_id), group in tqdm(groups, total=groups.ngroups, desc=f"Extract {part_path.stem}"):
        paper_code, review_id = str(paper_code), str(review_id)
        if review_id in done:
            continue
        sent_dict = build_sentence_dict(group)
        raw, error = call_aspects_for_review(sent_dict, paper_code, review_id)
        if error is None:
            append_jsonl(results_path, dict(paper_id=paper_code, review_id=review_id, aspects=raw))
            done.add(review_id)
        else:
            append_jsonl(errors_path, dict(paper_id=paper_code, review_id=review_id,
                                           error=error, n_sentences=len(sent_dict),
                                           raw_response=raw[:4000] if isinstance(raw, str) else None))
        processed_total += 1
        if processed_total % SAVE_EVERY == 0:
            state_path.write_text(json.dumps(dict(part=part_path.stem,
                                                   processed_reviews_global=processed_total,
                                                   done_reviews_in_part=len(done)), indent=2), encoding="utf-8")
        if SLEEP_EACH_CALL:
            time.sleep(SLEEP_EACH_CALL)
print(f"Wrote aspect responses under {response_dir}")
